In [1]:

import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

In [2]:

print(tf.__version__)

2.15.0


In [3]:
data_train_path = r"C:\Users\Kanis\Downloads\Image_classification\Vechiles_dataset\train"
data_test_path  = r"C:\Users\Kanis\Downloads\Image_classification\Vechiles_dataset\test"
data_val_path   = r"C:\Users\Kanis\Downloads\Image_classification\Vechiles_dataset\val"

In [4]:
img_width = 180
img_height =180

In [5]:
data_train = tf.keras.utils.image_dataset_from_directory(
    data_train_path,
    shuffle=True,
    image_size=(img_width, img_height),
    batch_size=32,
    validation_split=False)

Found 4486 files belonging to 8 classes.


In [6]:
data_cat = data_train.class_names

In [7]:
data_cat

['Ambulances',
 'Auto Rickshaws',
 'Bikes',
 'Cars',
 'Motorcycles',
 'Planes',
 'Ships',
 'Trains']

In [8]:
data_val = tf.keras.utils.image_dataset_from_directory(data_val_path,
                                                       image_size=(img_height,img_width),
                                                       batch_size=32,
                                                        shuffle=False,
                                                       validation_split=False)

Found 3678 files belonging to 8 classes.


In [9]:
data_test = tf.keras.utils.image_dataset_from_directory(
data_test_path,
    image_size=(img_height,img_width),
    shuffle=False,
    batch_size=32,
    validation_split=False
)

Found 958 files belonging to 8 classes.


In [10]:
from tensorflow.keras.models import Sequential

In [11]:
model = Sequential([
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32,3, padding='same',activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.2),
    layers.Dense(128),
    layers.Dense(len(data_cat))

])

In [12]:
model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

In [13]:
epochs_size = 25
history = model.fit(data_train, validation_data=data_val, epochs=epochs_size)

Epoch 1/25


141/141 [==============================] - 35s 229ms/step - loss: 1.4971 - accuracy: 0.4804 - val_loss: 0.7697 - val_accuracy: 0.7523
Epoch 2/25
141/141 [==============================] - 30s 213ms/step - loss: 0.7374 - accuracy: 0.7506 - val_loss: 0.5068 - val_accuracy: 0.8412
Epoch 3/25
141/141 [==============================] - 31s 220ms/step - loss: 0.4896 - accuracy: 0.8366 - val_loss: 0.4189 - val_accuracy: 0.8662
Epoch 4/25
141/141 [==============================] - 30s 211ms/step - loss: 0.3340 - accuracy: 0.8870 - val_loss: 0.4659 - val_accuracy: 0.8613
Epoch 5/25
141/141 [==============================] - 30s 212ms/step - loss: 0.2244 - accuracy: 0.9220 - val_loss: 0.4258 - val_accuracy: 0.8820
Epoch 6/25
141/141 [==============================] - 30s 215ms/step - loss: 0.1664 - accuracy: 0.9474 - val_loss: 0.5424 - val_accuracy: 0.8573
Epoch 7/25
141/141 [==============================] - 31s 216ms/step - loss: 0.1293 - accuracy: 0.9550 - val_loss: 0.4328 - val_

In [14]:
image = r"C:/Users/Kanis/Downloads/AI_batch_hackthon/ship_images.jpg"
image = tf.keras.utils.load_img(image, target_size=(img_height,img_width))
/#img_arr = tf.keras.utils.array_to_img(image)
img_arr = tf.keras.utils.img_to_array(image)
img_bat=tf.expand_dims(img_arr,0)

In [15]:
predict = model.predict(img_bat)

1/1 [==============================] - 0s 185ms/step


In [16]:
score = tf.nn.softmax(predict)

In [17]:
output=data_cat[np.argmax(score)]

In [18]:
print(score)

tf.Tensor(
[[2.1951649e-11 2.2914793e-18 5.0247648e-14 4.3235540e-14 2.3990003e-14
  7.9112127e-04 9.9920887e-01 1.9945307e-15]], shape=(1, 8), dtype=float32)


In [19]:
print(output)
score=np.max(score)*100
print(score)

if score >= 0.80:
    decision = "High Confidence Classification"
elif score >= 0.60:
    decision = "Ambiguous / Needs Review"
else:
    decision = "Uncertain Prediction"

print(decision)

Ships
99.92088675498962
High Confidence Classification


In [20]:
model.save('hackthon_Image_classification_model.keras')

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix

In [22]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

test_ds = image_dataset_from_directory(
    data_test_path,
    image_size=(180, 180),
    batch_size=32,
    shuffle=False  # VERY IMPORTANT
)

Found 958 files belonging to 8 classes.


In [23]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)

In [24]:
y_pred_prob = model.predict(test_ds)
y_pred = np.argmax(y_pred_prob, axis=1)

30/30 [==============================] - 2s 71ms/step


In [25]:


print("Accuracy:", accuracy_score(y_true, y_pred))

Accuracy: 0.7567849686847599


In [26]:


print("Precision:", precision_score(y_true, y_pred,  average='macro'))
print("Recall:", recall_score(y_true, y_pred,  average='macro'))

Precision: 0.7647150379630661
Recall: 0.7570378151260504


In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.66      0.66       120
           1       0.69      0.82      0.75       120
           2       0.99      0.90      0.94       119
           3       0.65      0.49      0.56       120
           4       0.80      0.84      0.82       120
           5       0.74      0.86      0.80       119
           6       0.92      0.72      0.81       120
           7       0.66      0.77      0.71       120

    accuracy                           0.76       958
   macro avg       0.76      0.76      0.76       958
weighted avg       0.76      0.76      0.76       958

